# RBX-AI Studio en Google Colab (gratis, uso personal)

Este notebook monta tu servidor Node.js (RBX-AI Studio) en Colab y te da una URL pública con HTTPS para que tu plugin de Roblox Studio se conecte a ella.

**Cómo usar (3 pasos):**
1. Sube tu repo a GitHub.
2. En Colab: **File → Open notebook → GitHub** → pega la URL de tu repo y elige `colab_server.ipynb`.
3. Pulsa **Connect** (arriba a la derecha) y ejecuta la **Celda 1**. Cuando termine, verás la URL en azul: ábrela **en una pestaña nueva del navegador** (no dentro de Colab) y copia esa misma URL en la ventana del plugin en Roblox Studio.

**Importante:** al abrir la URL la primera vez puede aparecer una pantalla de Cloudflare pidiéndote escribir **allow**; hazlo y la URL quedará activa. Si Colab te ofrece abrir el puerto con su propio proxy, también vale: es la opción más estable de todas.

**Nota:** la sesión dura hasta ~12 h. Al cerrarse, vuelve a ejecutar la Celda 1 (la URL cambia, actualízala en el plugin). Roblox Studio y el servidor hablan con el plugin por polling cada 0.5 s: mantén la pestaña de Colab abierta mientras programas.

In [ ]:
# Celda 1: descargar el proyecto, arrancar el servidor y crear el túnel público
import os, subprocess, threading, time
from google.colab import output

# ── 1. Clona tu repo de GitHub ───────────────────────────────────────────
# Pon aquí la URL de tu repo (o la URL raw de los archivos, más abajo)
REPO = "https://github.com/TU_USUARIO/TU_REPO.git"            # <--- TU REPO
FILES_PATH = "IA programacion roblox studio"                   # carpeta dentro del repo

os.chdir("/content")
if not os.path.exists("rbxai/server.js"):
    # Intenta clonar el repo entero
    !git clone --depth 1 "$REPO" rbxai 2>/dev/null
    if not os.path.exists("rbxai/server.js"):
        os.makedirs("rbxai", exist_ok=True)
        # Fallback: descarga los archivos sueltos del repo
        base = "https://raw.githubusercontent.com/TU_USUARIO/TU_REPO/main/" + FILES_PATH + "/"  # <--- TU RUTA
        for f in ["server.js", "index.html", "package.json", "package-lock.json"]:
            !wget -q -O "rbxai/$f" "$base$f"

# ── 2. Tu API key de OpenRouter (la escribes tú, nunca se guarda) ────────
api_key = output.eval_js("prompt('Pega tu OPENROUTER_API_KEY de https://openrouter.ai/keys')")
if not api_key or api_key == "null":
    raise RuntimeError("Sin API key no puedo arrancar. Consíguela gratis en https://openrouter.ai/keys")
os.environ["OPENROUTER_API_KEY"] = api_key
PORT = "8080"
os.environ["PORT"] = PORT

# ── 3. Instalar dependencias ─────────────────────────────────────────────
os.chdir("rbxai")
os.system("npm install --silent")

# ── 4. Arrancar el servidor Node.js (escuchando en 0.0.0.0, NO solo 127.0.0.1) ──
proc = subprocess.Popen(
    ["node", "server.js"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    env=os.environ
)
def tail():
    for line in proc.stdout:
        print(line.decode(), end="")
threading.Thread(target=tail, daemon=True).start()
time.sleep(8)
print("✅ Servidor Node.js corriendo en el puerto " + PORT)

# ── 5. Opción A (RECOMENDADA): proxy oficial de Colab, sin túneles externos ──
# Colab expone el puerto directamente con HTTPS y dominio propio de Google.
# Es la más estable: no depende de Cloudflare ni ngrok.
try:
    output.serve_kernel_port_as_window(int(PORT))
    print("🔗 OPCIÓN A — URL del proxy de Colab (la más estable):")
    print(f"   https://colab.research.google.com/drive/ -> Abre el menú 'Puertos' y pulsa el icono de globo junto al 8080")
except Exception as e:
    print("(proxy de Colab no disponible: " + str(e) + ")")

# ── 6. Opción B: túnel Cloudflare (quick tunnel, sin registro) ───────────
# FIX: usamos http://127.0.0.1 para evitar que cloudflared resuelva localhost
# a ::1 (IPv6) y devuelva 502 permanente.
os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")

tun = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:" + PORT, "--protocol", "quic"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
def tail_tunnel():
    for line in tun.stdout:
        print(line.decode(), end="")
threading.Thread(target=tail_tunnel, daemon=True).start()

# Esperar a que la URL aparezca en el log (puede tardar 15-40 s la primera vez).
# Cloudflare escribe la URL en stdout: redirigimos la salida del túnel a un archivo.
print("⏳ Creando túnel público (puede tardar hasta 60 s)...")
import re

logfile = "/content/cloudflared.log"
tun.stdout = open(logfile, "a")
tun.stderr = open(logfile, "a")
time.sleep(25)  # dejar que el túnel registre su URL

url = None
try:
    log_text = open(logfile).read()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
    if m:
        url = m.group(0)
except Exception:
    pass

# Verificar que la URL está viva (al primer registro puede tardar 30-60 s en responder 200)
if url:
    import urllib.request
    ok = False
    for i in range(24):
        time.sleep(5)
        try:
            code = urllib.request.urlopen(url, timeout=15).getcode()
            if code == 200:
                ok = True
                break
        except Exception:
            pass
    print()
    print("=" * 60)
    if ok:
        print("🔗 OPCIÓN B — Abre esta URL en una pestaña NUEVA del navegador:")
        print("   " + url)
        print("   (Si Cloudflare te pide escribir 'allow', hazlo: solo pasa la primera vez)")
        print("   Pégalas también en la ventana 'RBX-AI Bridge' del plugin en Studio")
    else:
        print("⚠️  La URL del túnel aún no responde. Espera 1 minuto más y revisa")
        print("   el log de arriba. Prueba usar la OPCIÓN A (proxy de Colab).")
    print("=" * 60)
else:
    print("⚠️  No se detectó la URL del túnel aún. Revisa el log de cloudflared")
    print("   arriba y usa la OPCIÓN A (proxy de Colab) si sigue fallando.")